# 07 — Evaluate the optimized candidate on holdout data

**Foundry feature:** the check the wizard's own reported score can't give you — generalization,
measured on `dataset/holdout.jsonl`, a split the optimizer never saw, plus whether a score delta is
even real given optimizer run-to-run noise. **Mode: CLI.**

This is the notebook `../prompt-agent-optimizer-baselines/_tools/build_foundry_dataset.py` names by
this exact filename in its own docstring: "the holdout file is consumed exclusively by the
post-optimization scoring notebook (`07_evaluate_optimized_candidate.ipynb`), never by the wizard."

In [ ]:
import json, subprocess, sys
from pathlib import Path

# Repo layout: this notebook lives in notebooks/, the pack lives in ../prompt-agent-optimizer-baselines
PACK_ROOT = Path("..").resolve() / "prompt-agent-optimizer-baselines"
AGENT_ID = "01-travel-approval-strict"          # <- the one case study every notebook in this series uses
AGENT_DIR = PACK_ROOT / AGENT_ID

assert AGENT_DIR.exists(), f"Can't find {AGENT_DIR} -- run this notebook from a checkout of the repo."

def run(cmd, cwd=PACK_ROOT):
    """Run a pack CLI tool and print its output, the way you would from a terminal."""
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    return result

print(f"Pack root : {PACK_ROOT}")
print(f"Case study: {AGENT_ID}")

## Why the wizard's own composite score isn't the final answer

The Optimize wizard reports a composite score computed **in-sample**, against the dataset you
uploaded (`dataset/optimize.jsonl`). That number tells you how well the candidate fits the data the
optimizer got to see. It says nothing about generalization on its own — which is exactly the gap
`dataset/holdout.jsonl` exists to close.

## Gating tests, split by which data they run against

Each agent's `expected/expectations.json` defines concrete `agent_tests`: an input, a required
behavior, and (for most of them) `regression_blocks: true`, meaning a failure here blocks promotion
regardless of composite score. Roughly a third of this case study's tests are reserved for holdout.

In [ ]:
expectations = json.loads((AGENT_DIR / "expected" / "expectations.json").read_text())
tests = expectations["agent_tests"]
by_source = {"optimize": [], "holdout": []}
for t in tests:
    by_source[t["dataset_source"]].append(t)

for source, ts in by_source.items():
    print(f"{source}: {len(ts)} tests, {sum(t.get('regression_blocks', False) for t in ts)} gating")
    for t in ts:
        print(f"  [{t['id']}] {t['input'][:70]}...  (regression_blocks={t.get('regression_blocks', False)})")

A pass on `dataset/optimize.jsonl` alone is weaker evidence than a pass on `dataset/holdout.jsonl`
— the optimizer had the chance to fit directly to the optimize-split inputs. Treat a holdout pass
as the real evidence of generalization.

> **Response-level harness gap, stated plainly.** Actually *running* these gating tests requires
> calling the deployed, optimized agent with each test's `input` and checking its real response and
> tool calls — a response-level harness this pack doesn't ship yet (see
> `../prompt-agent-optimizer-baselines/_tools/run_manifest_template.json`'s
> `holdout_composite_score_note`). If you have portal or API access to the optimized agent, wire that
> call in below; this notebook shows you exactly what to check once you have a response.

In [ ]:
def check_response(test: dict, response: str, tool_calls: list) -> dict:
    """Minimal response-level check against one agent_tests entry's `expect` block."""
    expect = test["expect"]
    ok = True
    reasons = []
    for phrase in expect.get("must_contain", []):
        if phrase.lower() not in response.lower():
            ok = False
            reasons.append(f"missing required phrase: {phrase!r}")
    policy = expect.get("tool_call", {}).get("policy")
    required_tools = set(expect.get("tool_call", {}).get("tools", []))
    called = set(tool_calls)
    if policy == "required" and not required_tools.issubset(called):
        ok = False
        reasons.append(f"required tool(s) not called: {required_tools - called}")
    if policy == "forbidden" and required_tools & called:
        ok = False
        reasons.append(f"forbidden tool(s) called: {required_tools & called}")
    return {"id": test["id"], "pass": ok, "reasons": reasons}

# Example call, once you have a real response from the deployed candidate:
# check_response(by_source["holdout"][0], response="...the agent's real reply...", tool_calls=["lookup_travel_policy"])
print("Helper defined -- plug in real (response, tool_calls) pairs from your deployed candidate.")

## Replicate seeds — one run is one draw, not a result

A single optimization run is one draw from a stochastic search process. How many replicate runs you
need depends on what you intend to claim:

- **k ≥ 3** gives a descriptive estimate (mean, standard deviation, bootstrap confidence interval).
- **k ≥ 9** is the minimum for a paired Wilcoxon signed-rank test to be able to reach significance at
  all after Holm-Bonferroni correction — its smallest achievable two-sided p-value at k pairs is
  `2^(1-k)`, so at k=5 the floor is 0.0625: no effect size, however large, can produce p < 0.05 at
  that sample size. Run k = 10 if you intend to report a significance claim.

Save each replicate run's exported candidate under a distinct name (`foundry_run1.md`,
`foundry_run2.md`, ...) and a matching `run_manifest.json` (notebook 04). This cell aggregates
whatever manifests already exist for this case study under `runs/<agent_id>/`.

In [ ]:
import statistics, random

runs_dir = PACK_ROOT / "runs" / AGENT_ID
manifests = sorted(runs_dir.glob("*.manifest.json")) if runs_dir.exists() else []
print(f"{len(manifests)} run manifest(s) found for {AGENT_ID}:")

scores = []
for m in manifests:
    data = json.loads(m.read_text())
    holdout_score = data["outputs"].get("holdout_composite_score")
    print(f"  {m.name}: run_label={data.get('run_label')}  holdout_composite_score={holdout_score}")
    if holdout_score is not None:
        scores.append(holdout_score)

if len(scores) >= 3:
    mean = statistics.mean(scores)
    stdev = statistics.stdev(scores)
    boot = [statistics.mean(random.choices(scores, k=len(scores))) for _ in range(2000)]
    boot.sort()
    lo, hi = boot[int(0.025 * len(boot))], boot[int(0.975 * len(boot))]
    print(f"\nk={len(scores)}  mean={mean:.3f}  sd={stdev:.3f}  95% bootstrap CI=({lo:.3f}, {hi:.3f})")
    if len(scores) < 9:
        print("k < 9: descriptive only -- do not report a Holm-corrected significance claim at this sample size.")
else:
    print("\nFewer than 3 replicate holdout scores recorded yet -- fill in "
          "outputs.holdout_composite_score in each run's manifest once you have real holdout evaluations.")

## Next

Continue to **`08_compare_to_open_baseline.ipynb`** to see how this same case study performs against
an open, reproducible optimizer alternative to Foundry's closed preview.